# M4: Cosmos Transfer — Weather Augmentation

**Stage 5: Data Augmentation — Weather Variants (Cosmos Transfer 2.5)**

| Item | Detail |
|------|--------|
| Model | Cosmos-Transfer2.5-2B (`general/edge` control) |
| Instance | Adapts to your GPU: **g5/g6.12xlarge+** run 480p (multi-GPU shard); **p4d.24xlarge** (A100) / **p5.48xlarge** (H100) run full 720p. The notebook auto-detects and picks the strategy. |
| Input | nuScenes CAM_FRONT frames selected in **M1** (`m1/manifest.json`) |
| Output | `/users/{profile}/m4/` (weather-augmented driving clips) |
| Repo | [nvidia-cosmos/cosmos-transfer2.5](https://github.com/nvidia-cosmos/cosmos-transfer2.5) |

This module applies **edge-conditioned domain transfer** to a real driving clip
assembled from the nuScenes CAM_FRONT frames M1 selected. Cosmos Transfer 2.5
extracts a Canny **edge** control video on the fly and regenerates the scene
under a new weather prompt (rain / fog / night), preserving scene geometry while
changing appearance. This produces training data for conditions underrepresented
in the source dataset.

### How this notebook actually runs Cosmos Transfer

There is **no `pip install cosmos-transfer2`** — the real workflow is to clone
the official repo, `uv sync` its pinned deps (torch 2.7 + cu128, Python 3.10,
transformer-engine, megatron), and call `examples/inference.py`. All of that,
plus the SMD-image environment fixes it needs (opencv-headless, CUDA `.so`
symlinks, `CUDA_HOME`/`LD_LIBRARY_PATH`, HF xet disable), is encapsulated in
**`scripts/setup_cosmos_env.sh`**, which the first code cell runs for you.

> **Gated models:** the Cosmos checkpoints require a HuggingFace token whose
> account has accepted the licenses on
> [Cosmos-Guardrail1](https://huggingface.co/nvidia/Cosmos-Guardrail1),
> [Cosmos-Transfer2.5-2B](https://huggingface.co/nvidia/Cosmos-Transfer2.5-2B),
> and [Cosmos-Reason1-7B](https://huggingface.co/nvidia/Cosmos-Reason1-7B).
> Set `HF_TOKEN` in the setup cell below before running.

> **Ephemeral install:** the environment lives on the instance's local NVMe and
> is reset when the JupyterLab app restarts. Re-run the setup cell after any
> restart — it is idempotent and skips work already done.

In [ ]:
# ============================================================
# Setup — configuration
# ============================================================
import os
import sys
import time
import json
import glob
import subprocess
from pathlib import Path
from datetime import datetime, timezone

import boto3

# ------------------------------------------------------------
# HuggingFace token — usually NOT needed.
# The admin pre-caches the Cosmos checkpoints to S3, and the setup cell restores
# them and runs Hugging Face in OFFLINE mode — so most participants leave this
# blank. Only paste a token if your admin says the S3 cache is unavailable and
# you must download the gated models yourself (then also accept their licenses).
# An env var (admin-injected) always takes priority.
# ------------------------------------------------------------
HF_TOKEN = os.environ.get("HF_TOKEN", "") or ""   # optional; leave "" to use the offline cache
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]
PROFILE = os.environ.get("USER_PROFILE", os.environ.get("BLUEPRINT_PROFILE", "default"))
SHARED_BUCKET = os.environ.get("SHARED_BUCKET", f"av30lab-shared-data-{ACCOUNT_ID}")
S3_BUCKET = os.environ.get("USER_BUCKET", f"av30lab-user-workspace-{ACCOUNT_ID}")

# M1 selected the nuScenes CAM_FRONT frames; its manifest lists their S3 keys.
M1_PREFIX = f"users/{PROFILE}/m1/"
OUTPUT_PREFIX = f"users/{PROFILE}/m4/"
NUSCENES_PREFIX = "datasets/nuscenes-mini/"   # in SHARED_BUCKET

# Local scratch on the instance NVMe (28 TB on p4d/p5; models are large).
NVME = "/mnt/sagemaker-nvme" if os.path.isdir("/mnt/sagemaker-nvme") else "/tmp"
WORK = f"{NVME}/m4_work"
FRAMES_DIR = f"{WORK}/frames"
INPUT_MP4 = f"{WORK}/nuscenes_cam_front.mp4"
OUTPUT_DIR = f"{WORK}/out"

# Cosmos Transfer env produced by scripts/setup_cosmos_env.sh
COSMOS_WORK = f"{NVME}/cosmos-work"
COSMOS_REPO = f"{COSMOS_WORK}/cosmos-transfer2.5"
COSMOS_ENV_FILE = f"{COSMOS_WORK}/cosmos_env.sh"

# Video assembly params (validated: 57 frames @ 1280x704 works end-to-end).
VIDEO_W, VIDEO_H = 1280, 704
VIDEO_FPS = 10
MAX_FRAMES = 57
GUIDANCE = 5   # 0..7 — higher = follow the weather prompt more closely

# Weather variants to generate. Each becomes one Cosmos Transfer run.
WEATHER_PROMPTS = {
    "rain": (
        "A realistic driving scene in heavy rain. Wet reflective road surfaces, "
        "water droplets, spray from tires, overcast sky, reduced visibility. The "
        "scene structure, road layout and vehicles stay identical to the input; "
        "only the weather becomes rainy."
    ),
    "fog": (
        "A realistic driving scene in dense fog. Severely reduced visibility, "
        "diffused headlights, hazy grey atmosphere, distant objects fading into "
        "mist. Scene geometry and vehicles are unchanged; only the weather "
        "becomes foggy."
    ),
    "night": (
        "A realistic driving scene at night. Dark sky, headlight and streetlight "
        "illumination, high contrast between lit and unlit areas, glare on wet "
        "asphalt. Scene structure and vehicles are unchanged; only the time of "
        "day becomes night."
    ),
}
# Keep the workshop run cheap/fast by default: one variant. Set to the full list
# to generate all three.
CONDITIONS = ["rain"]

os.makedirs(FRAMES_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Profile: {PROFILE}")
print(f"Input  : nuScenes CAM_FRONT via {M1_PREFIX}manifest.json")
print(f"Output : s3://{S3_BUCKET}/{OUTPUT_PREFIX}")
print(f"Work   : {WORK}")
print(f"Weather conditions: {CONDITIONS}")
print(f"HF_TOKEN: {'set (online fallback)' if HF_TOKEN else 'not set (using admin offline cache — normal)'}")

In [ ]:
# ============================================================
# Pre-flight — detect GPUs and choose an instance-appropriate run strategy
# ============================================================
# Cosmos-Transfer2.5-2B needs ~65 GB to load on ONE GPU. Almost no workshop GPU
# has that on a single card, so the RIGHT strategy depends on the instance you
# launched. We detect PER-GPU VRAM (not the sum) and pick:
#
#   per-GPU >= 70 GB (H100 / p5)   -> single GPU,        720p, guardrails ON
#   per-GPU >= 38 GB (A100 / p4d)  -> N-GPU shard,       720p, guardrails ON
#   per-GPU  < 38 GB (L4/A10G,g5/g6)-> N-GPU shard,      480p, guardrails OFF
#
# The N-GPU path uses torchrun context parallelism (shards the diffusion
# activations across ranks). On 24 GB cards we also drop to 480p and disable the
# guardrail models (Cosmos-Guardrail1 + Reason-7B). This path completes on a
# g6.24xlarge (4x L4 24GB) ONLY when the GPUs are otherwise idle — see the
# foreign-occupancy halt below. Bigger cards keep full 720p + guardrails ON.
#
# IMPORTANT: we probe the GPUs with `nvidia-smi` in a SUBPROCESS, never with
# torch.cuda in this kernel. Touching torch.cuda here creates a CUDA context
# INSIDE the JupyterLab kernel that lives for the whole session, permanently
# pinning GPU memory the torchrun workers then need — a real cause of the VAE
# tokenizer OOM. nvidia-smi reads the driver without a CUDA context, so the
# kernel stays off the GPUs entirely.
import subprocess

FOREIGN_MIB = 1024   # >1 GiB used, with no run in flight, == a foreign process

def _probe_gpus_nvidia_smi():
    """Per-GPU (total_mib, used_mib) via nvidia-smi — creates NO CUDA context."""
    out = subprocess.run(
        ["nvidia-smi", "--query-gpu=memory.total,memory.used",
         "--format=csv,noheader,nounits"],
        capture_output=True, text=True, check=True,
    ).stdout
    gpus = []
    for line in out.strip().splitlines():
        if not line.strip():
            continue
        total_s, used_s = (p.strip() for p in line.split(","))
        gpus.append((int(total_s), int(used_s)))   # MiB, MiB
    return gpus

try:
    _gpus = _probe_gpus_nvidia_smi()
except FileNotFoundError:
    # nvidia-smi absent almost always means the CPU image was launched. Fall back
    # to torch ONLY to produce total VRAM so tiering still works; this DOES create
    # a kernel CUDA context (the thing we avoid above), but a card with no
    # nvidia-smi is not the multi-GPU shard path where that context matters.
    try:
        import torch
        assert torch.cuda.is_available()
        _n = torch.cuda.device_count()
        _gpus = [(int(torch.cuda.get_device_properties(i).total_memory / (1024**2)), 0)
                 for i in range(_n)]
    except Exception as e:
        raise RuntimeError(
            "nvidia-smi not found and no CUDA GPU visible — this looks like the "
            "CPU image. Use Instance Options to pick a GPU instance, then re-run."
        ) from e
except subprocess.CalledProcessError as e:
    raise RuntimeError(
        "nvidia-smi failed — this notebook requires a GPU instance. "
        "Use Instance Options to pick a GPU type."
    ) from e

assert _gpus, "nvidia-smi reported no GPUs — this notebook requires a GPU instance."

GPU_COUNT = len(_gpus)                               # rows == device count
per_gpu_gb = [tot / 1024 for tot, _ in _gpus]        # MiB -> GiB (same scale as torch)
per_gpu_used_mib = [used for _, used in _gpus]
MIN_PER_GPU_GB = min(per_gpu_gb)
print(f"GPUs available: {GPU_COUNT}")
for i, (tot, used) in enumerate(_gpus):
    print(f"  GPU {i}: {tot / 1024:.1f} GB total — {used} MiB in use")
print(f"Smallest GPU: {MIN_PER_GPU_GB:.1f} GB")

# --- Halt if ANOTHER process already holds the GPUs ------------------------
# The kernel never touches CUDA (we probe via nvidia-smi), so any significant
# 'in use' memory here belongs to a FOREIGN process — almost always another
# notebook kernel left running, or a stale torchrun rank from an interrupted
# run. Cosmos Transfer needs (nearly) the whole card, so leaving it there would
# OOM deep in the VAE tokenizer with an opaque CUDA error AFTER the ~15-20 min
# setup + model load. Detect it and stop cleanly now.
_busy = [(i, u) for i, u in enumerate(per_gpu_used_mib) if u > FOREIGN_MIB]
if _busy:
    _detail = ", ".join(f"GPU {i}: {u} MiB" for i, u in _busy)
    raise SystemExit(
        "\n"
        "==================================================================\n"
        " GPU MEMORY IS ALREADY IN USE BY ANOTHER PROCESS — cannot start.\n"
        "==================================================================\n"
        f"  Occupied now: {_detail}\n\n"
        "  This is almost always ANOTHER notebook kernel left running (or a\n"
        "  stale worker from an interrupted run). Cosmos Transfer needs nearly\n"
        f"  the whole GPU (~{MIN_PER_GPU_GB:.0f} GB per card), so this run would\n"
        "  crash with a CUDA out-of-memory error deep inside the model — NOT\n"
        "  something you can fix by editing this notebook.\n\n"
        "  TO FIX — free the GPUs, then Run All Cells again:\n"
        "    1. JupyterLab top menu:  Kernel -> Shut Down All Kernels\n"
        "       (or the left sidebar 'Running Terminals and Kernels' tab ->\n"
        "        'Shut Down All' under KERNELS).\n"
        "    2. Confirm the cards are free — in a terminal run:\n"
        "         nvidia-smi --query-gpu=memory.used --format=csv,noheader\n"
        "       every GPU should read a few MiB (near 0).\n"
        "    3. Then Run All Cells again (Run menu) — NOT just this cell. A kernel shutdown clears the setup cell above (where MAX_FRAMES etc. are defined), so re-running only this cell fails; Run All re-creates what the guard needs.\n"
        "=================================================================="
    )

# --- Choose the run strategy from per-GPU VRAM -----------------------------
if MIN_PER_GPU_GB >= 70:          # H100 80GB (p5)
    RUN_MODE = "single"           # model fits on one GPU
    RUN_RESOLUTION = "720"
    USE_GUARDRAILS = True
    RUN_MAX_FRAMES = MAX_FRAMES   # full clip
    _why = "H100-class GPU: full 720p on a single GPU with guardrails on."
elif MIN_PER_GPU_GB >= 38:        # A100 40GB (p4d)
    RUN_MODE = "shard"            # torchrun context-parallel across all GPUs
    RUN_RESOLUTION = "720"
    USE_GUARDRAILS = True
    RUN_MAX_FRAMES = MAX_FRAMES   # full clip
    _why = f"A100-class GPUs: full 720p, sharded across {GPU_COUNT} GPUs, guardrails on."
else:                             # L4/A10G 24GB (g5/g6) — verified path
    RUN_MODE = "shard"
    RUN_RESOLUTION = "480"
    USE_GUARDRAILS = False
    # Cap inference frames on 24GB cards: fewer frames = less tokenizer activation
    # memory. 16 was verified to complete on g6.24xlarge (4x L4 24GB) with the
    # GPUs otherwise idle (no other notebook kernel holding memory).
    RUN_MAX_FRAMES = min(MAX_FRAMES, 16)
    _why = (f"24GB-class GPUs: 480p, {min(MAX_FRAMES,16)} frames, sharded across "
            f"{GPU_COUNT} GPUs, guardrails OFF (the guardrail models don't fit "
            f"alongside the pipeline here). For full 720p / all {MAX_FRAMES} frames, "
            f"relaunch on ml.p4d.24xlarge (A100 40GB) or ml.p5.48xlarge (H100 80GB).")

if RUN_MODE == "shard" and GPU_COUNT < 2:
    raise RuntimeError(
        f"This GPU has only {MIN_PER_GPU_GB:.0f} GB and there is just {GPU_COUNT} GPU, "
        "so Cosmos Transfer 2.5-2B cannot be sharded to fit. Pick a multi-GPU "
        "instance (g5.12xlarge / g6.12xlarge / p4d.24xlarge) in Instance Options."
    )

print(f"\nRun strategy: mode={RUN_MODE}, resolution={RUN_RESOLUTION}p, "
      f"frames={RUN_MAX_FRAMES}, guardrails={'on' if USE_GUARDRAILS else 'off'}")
print(f"  -> {_why}")
print("GPU check PASSED.")

In [ ]:
# ============================================================
# Install the Cosmos Transfer 2.5 environment (idempotent)
# ============================================================
# Runs scripts/setup_cosmos_env.sh, which:
#   - clones the official repo + uv-syncs the pinned CUDA stack,
#   - applies the SMD-image fixes (opencv-headless, CUDA .so symlinks, ldconfig,
#     CUDA_HOME/LD_LIBRARY_PATH written to cosmos_env.sh),
#   - RESTORES the admin's pre-cached Cosmos checkpoints from S3 into HF_HOME and
#     sets HF_HUB_OFFLINE=1 (so inference needs NO Hugging Face token),
#   - if that S3 cache is absent, falls back to online download (needs HF_TOKEN).
# Re-running after an app restart is safe. First run: ~15-20 min.

# No hard token requirement — the offline cache is the normal path. If neither
# the cache nor a token is available, the inference cell fails with a clear HF
# "gated / offline" error and you can set HF_TOKEN in the config cell.
if HF_TOKEN:
    print("HF_TOKEN provided — online download available as fallback.")
else:
    print("No HF_TOKEN — relying on the admin's offline S3 cache (normal path).")

# Locate the setup script (repo layout: notebooks/ sits next to scripts/). Guard
# each probe with try/except: some candidate dirs (e.g. /root) raise
# PermissionError from exists() for a non-root user.
def _find_setup_script():
    for base in [Path.cwd(), Path.cwd().parent, Path.home()]:
        cand = base / "scripts" / "setup_cosmos_env.sh"
        try:
            if cand.exists():
                return str(cand)
        except OSError:
            continue
    return None

setup_script = _find_setup_script()
if setup_script is None:
    # Fall back to a copy staged in S3 (notebook-templates ships scripts/ too).
    local = f"{WORK}/setup_cosmos_env.sh"
    subprocess.run(
        ["aws", "s3", "cp",
         f"s3://{SHARED_BUCKET}/notebook-templates/scripts/setup_cosmos_env.sh", local],
        check=True,
    )
    setup_script = local

print(f"Running setup script: {setup_script}")
print("(first run: 15-20 min for uv sync + HF cache restore; re-runs are fast)\n")

env = {**os.environ, "SHARED_BUCKET": SHARED_BUCKET}
if HF_TOKEN:
    env["HF_TOKEN"] = HF_TOKEN
proc = subprocess.run(["bash", setup_script], env=env, text=True)
if proc.returncode != 0:
    raise RuntimeError(
        "setup_cosmos_env.sh failed — scroll up for the error. Common causes: no "
        "S3 HF cache AND no HF_TOKEN, or not enough NVMe space."
    )
print("\nCosmos Transfer 2.5 environment ready.")

In [ ]:
# ============================================================
# Build the input driving clip from M1's nuScenes CAM_FRONT frames
# ============================================================
# M1 wrote users/{profile}/m1/manifest.json listing the CAM_FRONT frame keys it
# selected (relative to the shared nuScenes dataset). We download those frames
# and stitch them, in order, into a single mp4 that Cosmos Transfer will use as
# its source (it computes the Canny edge control video from this on the fly).
#
# NOTE: this cell runs in the JupyterLab KERNEL (SMD python 3.12), which does
# NOT ship cv2. (Only the repo's uv venv, used by the inference cell, has it.)
# Install the headless build here (no libGL / system GL libs needed). Use
# --no-deps so pip does NOT pull a newer numpy and break the SMD image's many
# numpy<2.4 packages (autogluon, sagemaker-studio, ...); SMD already has numpy.
try:
    import cv2
except ModuleNotFoundError:
    print("Installing opencv-python-headless (--no-deps) into the notebook kernel...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--no-deps",
         "opencv-python-headless"],
        check=True,
    )
    import cv2
print(f"cv2 {cv2.__version__} ready in kernel")

# 1) Read M1's manifest to learn which CAM_FRONT frames to use.
m1_manifest_key = f"{M1_PREFIX}manifest.json"
print(f"Reading M1 manifest: s3://{S3_BUCKET}/{m1_manifest_key}")
s3 = boto3.client("s3")
try:
    body = s3.get_object(Bucket=S3_BUCKET, Key=m1_manifest_key)["Body"].read()
    m1_manifest = json.loads(body)
    cam_front_files = m1_manifest.get("cam_front_files", [])
except s3.exceptions.NoSuchKey:
    raise RuntimeError(f"M1 manifest not found — run M1 first ({m1_manifest_key})")

print(f"M1 selected {len(cam_front_files)} CAM_FRONT frames")
assert cam_front_files, "M1 manifest has no cam_front_files — re-run M1"

# 2) Download the frames from the shared nuScenes dataset (they live under
#    datasets/nuscenes-mini/<filename>). Keep only up to MAX_FRAMES, in order.
frames_local = []
for rel in cam_front_files[:MAX_FRAMES]:
    key = f"{NUSCENES_PREFIX}{rel}"
    dest = os.path.join(FRAMES_DIR, os.path.basename(rel))
    if not os.path.exists(dest):
        s3.download_file(SHARED_BUCKET, key, dest)
    frames_local.append(dest)
print(f"Downloaded {len(frames_local)} frames to {FRAMES_DIR}")

# 3) Assemble into an mp4 (resized to the Cosmos-friendly 1280x704).
vw = cv2.VideoWriter(
    INPUT_MP4, cv2.VideoWriter_fourcc(*"mp4v"), VIDEO_FPS, (VIDEO_W, VIDEO_H)
)
n_written = 0
for f in frames_local:
    img = cv2.imread(f)
    if img is None:
        continue
    img = cv2.resize(img, (VIDEO_W, VIDEO_H))
    vw.write(img)
    n_written += 1
vw.release()

assert n_written > 0 and os.path.exists(INPUT_MP4), "Failed to assemble input mp4"
print(f"Assembled input clip: {INPUT_MP4}")
print(f"  {n_written} frames @ {VIDEO_FPS} fps, {VIDEO_W}x{VIDEO_H}, "
      f"{os.path.getsize(INPUT_MP4)/1e6:.2f} MB")

In [ ]:
# ============================================================
# Build Cosmos Transfer inference spec(s)
# ============================================================
# examples/inference.py takes a JSON "spec" per sample. For edge control we only
# need the source video_path — the edge control video is computed on the fly
# (no control_path required). One spec per weather condition.
specs_dir = f"{WORK}/specs"
os.makedirs(specs_dir, exist_ok=True)

spec_paths = []
for cond in CONDITIONS:
    prompt = WEATHER_PROMPTS[cond]
    prompt_path = f"{specs_dir}/{cond}_prompt.txt"
    with open(prompt_path, "w") as f:
        f.write(prompt)

    spec = {
        "name": f"nuscenes_{cond}",
        "prompt_path": prompt_path,
        "video_path": INPUT_MP4,
        "guidance": GUIDANCE,
        # control_path omitted -> Cosmos computes the Canny edge map on the fly.
        "edge": {"control_weight": 1.0},
    }
    spec_path = f"{specs_dir}/nuscenes_{cond}_spec.json"
    with open(spec_path, "w") as f:
        json.dump(spec, f, indent=2)
    spec_paths.append(spec_path)
    print(f"  {cond}: {spec_path}")

print(f"\nBuilt {len(spec_paths)} spec(s) for conditions: {CONDITIONS}")

In [ ]:
# ============================================================
# Run Cosmos Transfer edge-to-video inference (instance-adaptive)
# ============================================================
# Uses the strategy chosen in the pre-flight cell:
#   RUN_MODE       single | shard        (shard = torchrun context-parallel)
#   RUN_RESOLUTION "720" | "480"
#   USE_GUARDRAILS True | False
# The edge control video is computed on the fly from the input mp4.
generation_start = time.time()
results_manifest = []

def _free_gpus():
    """Reclaim GPU memory before launching inference.

    A failed/interrupted torchrun leaves some ranks alive, each holding ~16 GB
    of GPU memory. A later run then starts on a GPU that already looks full and
    OOMs — even though the code is fine (this is why one weather condition can
    succeed and the next fail). So before each launch we kill any stale Cosmos
    inference workers (matched by the repo path, so the JupyterLab kernel and
    other users' processes are never touched) and clear our own CUDA cache.
    """
    import subprocess as _sp
    for pat in ("cosmos-transfer2.5/examples/inference.py",
                "cosmos-predict2.5/examples/inference.py"):
        _sp.run(["pkill", "-9", "-f", pat], capture_output=True)
    time.sleep(3)
    try:
        import torch as _t
        if _t.cuda.is_available():
            _t.cuda.empty_cache()
    except Exception:
        pass

# Common flags appended to inference.py (setup options, provided on the CLI).
def _extra_flags():
    flags = ["--resolution", RUN_RESOLUTION, "--max-frames", str(RUN_MAX_FRAMES)]
    if not USE_GUARDRAILS:
        flags += ["--disable-guardrails"]
    return " ".join(flags)

for cond, spec_path in zip(CONDITIONS, spec_paths):
    _free_gpus()   # reclaim any GPU memory held by a previous/failed run
    print(f"\n=== Generating '{cond}'  (mode={RUN_MODE}, {RUN_RESOLUTION}p, "
          f"guardrails={'on' if USE_GUARDRAILS else 'off'}) ===")
    t0 = time.time()

    if RUN_MODE == "single":
        launcher = "python"
    else:
        # torchrun spawns one rank per GPU; context_parallel_size defaults to
        # WORLD_SIZE, so the model activations shard across all GPUs.
        launcher = f"torchrun --nproc_per_node={GPU_COUNT} --master_port=12356"

    # expandable_segments reduces CUDA fragmentation (helps the 24GB path).
    inner = (
        f'export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True && '
        f'source "{COSMOS_ENV_FILE}" && cd "{COSMOS_REPO}" && '
        f'{launcher} examples/inference.py -i "{spec_path}" -o "{OUTPUT_DIR}" '
        f'{_extra_flags()} control:edge'
    )
    proc = subprocess.run(["bash", "-lc", inner], text=True, capture_output=True)
    tail = "\n".join((proc.stdout or "").splitlines()[-15:])
    print(tail)
    if proc.returncode != 0:
        err_tail = "\n".join((proc.stderr or "").splitlines()[-25:])
        raise RuntimeError(
            f"Cosmos Transfer failed for '{cond}':\n{err_tail}\n\n"
            f"If this is a CUDA out-of-memory on a 24GB GPU, the 480p+shard path "
            f"should fit — check that all {GPU_COUNT} GPUs were used. For 720p, "
            f"relaunch on p4d.24xlarge / p5.48xlarge."
        )

    elapsed = time.time() - t0
    gen_mp4 = os.path.join(OUTPUT_DIR, f"nuscenes_{cond}.mp4")
    edge_mp4 = os.path.join(OUTPUT_DIR, f"nuscenes_{cond}_control_edge.mp4")
    assert os.path.exists(gen_mp4), f"Expected output not found: {gen_mp4}"

    print(f"  -> {gen_mp4} ({os.path.getsize(gen_mp4)/1e6:.2f} MB) in {elapsed:.0f}s")
    results_manifest.append({
        "condition": cond,
        "source": "nuscenes_cam_front",
        "generated_video": os.path.basename(gen_mp4),
        "edge_control_video": os.path.basename(edge_mp4) if os.path.exists(edge_mp4) else None,
        "guidance": GUIDANCE,
        "resolution": RUN_RESOLUTION,
        "run_mode": RUN_MODE,
        "guardrails": USE_GUARDRAILS,
        "generation_time_s": round(elapsed, 1),
    })

generation_elapsed = time.time() - generation_start
print(f"\nGeneration complete: {len(results_manifest)} clip(s) in {generation_elapsed:.0f}s")

In [ ]:
# ============================================================
# Upload Results to S3
# ============================================================
output_s3_path = f"s3://{S3_BUCKET}/{OUTPUT_PREFIX}"
print(f"Uploading augmented clips to {output_s3_path}...")

# Sync the whole output dir (generated + edge-control mp4s + Cosmos config/logs).
result = subprocess.run(
    ["aws", "s3", "sync", OUTPUT_DIR, output_s3_path, "--quiet"],
    capture_output=True, text=True,
)
if result.returncode != 0:
    raise RuntimeError(f"Upload failed: {result.stderr}")

# Also keep the source clip we built, so the augmentation is reproducible.
subprocess.run(
    ["aws", "s3", "cp", INPUT_MP4, f"{output_s3_path}source/nuscenes_cam_front.mp4",
     "--quiet"],
    capture_output=True, text=True,
)

# Write + upload a manifest describing what was generated.
manifest = {
    "module": "M4_Cosmos_Transfer_Augmentation",
    "profile": PROFILE,
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "model": "Cosmos-Transfer2.5-2B (general/edge)",
    "control": "edge (Canny, computed on the fly)",
    "source": "nuScenes CAM_FRONT (frames selected by M1)",
    "conditions": CONDITIONS,
    "clips_generated": len(results_manifest),
    "results": results_manifest,
}
manifest_path = os.path.join(OUTPUT_DIR, "manifest.json")
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)
subprocess.run(
    ["aws", "s3", "cp", manifest_path, f"{output_s3_path}manifest.json", "--quiet"],
    capture_output=True, text=True,
)
print(f"Uploaded {len(results_manifest)} clip(s) + source + manifest")

In [ ]:
# ============================================================
# Cost Analysis
# ============================================================
# Reads the ACTUAL instance type from the SageMaker resource metadata so the
# estimate matches whatever GPU box you launched (g5/g6/p4d/p5).
INSTANCE_RATES = {
    "ml.g5.12xlarge": 7.09, "ml.g5.24xlarge": 10.18, "ml.g5.48xlarge": 20.36,
    "ml.g6.12xlarge": 5.53, "ml.g6.24xlarge": 9.84, "ml.g6.48xlarge": 19.69,
    "ml.p4d.24xlarge": 37.69, "ml.p5.48xlarge": 113.14,
}
USD_TO_KRW = 1370

inst = "unknown"
try:
    md = json.loads(Path("/opt/ml/metadata/resource-metadata.json").read_text())
    inst = md.get("InstanceType", "unknown")
except Exception:
    pass
rate = INSTANCE_RATES.get(inst)

gen_seconds = generation_elapsed
hours = gen_seconds / 3600

print("=" * 50)
print("COST ANALYSIS — M4 Cosmos Transfer Augmentation")
print("=" * 50)
print(f"Instance:        {inst}")
print(f"Run strategy:    {RUN_MODE}, {RUN_RESOLUTION}p, {RUN_MAX_FRAMES} frames, "
      f"guardrails={'on' if USE_GUARDRAILS else 'off'}")
print(f"Generation time: {gen_seconds:.0f}s ({hours:.3f} hr)")
if rate is not None:
    cost_usd = hours * rate
    print(f"Rate:            ${rate}/hr")
    print(f"Estimated cost:  ${cost_usd:.2f} USD / {cost_usd * USD_TO_KRW:,.0f} KRW")
    print(f"Clips produced:  {len(results_manifest)}")
    if results_manifest:
        print(f"Cost per clip:   ${cost_usd / len(results_manifest):.3f} USD")
else:
    print(f"Rate:            (unknown instance '{inst}' — not in rate table)")
    print(f"Clips produced:  {len(results_manifest)}")
print("=" * 50)
print("Note: the one-time environment install + checkpoint download (setup cell)")
print("adds ~15-25 min of instance time on the FIRST run of a fresh app.")
print("Tip: 24GB GPUs (g5/g6) run 480p; p4d/p5 run full 720p — see the pre-flight cell.")

In [ ]:
# ============================================================
# Output Validation + Next Module
# ============================================================
gen_files = [
    os.path.join(OUTPUT_DIR, r["generated_video"])
    for r in results_manifest
]
print("Output validation:")
print(f"  Generated clips: {len(gen_files)} (expected {len(CONDITIONS)})")

ok = True
for f in gen_files:
    exists = os.path.exists(f)
    size_mb = os.path.getsize(f) / 1e6 if exists else 0
    flag = "OK" if (exists and size_mb > 0.1) else "MISSING/TOO SMALL"
    print(f"    {os.path.basename(f)}: {size_mb:.2f} MB [{flag}]")
    ok = ok and exists and size_mb > 0.1

print(f"  Status: {'PASS' if ok and gen_files else 'FAIL'}")
print(f"\nOutput location: {output_s3_path}")

# Preview the first generated clip inline (optional).
try:
    from IPython.display import Video, display
    if gen_files and os.path.exists(gen_files[0]):
        display(Video(gen_files[0], embed=True, width=640))
except Exception as e:
    print(f"(inline preview skipped: {e})")

print("\n" + "=" * 50)
print("NEXT MODULE")
print("=" * 50)
print("M5: Cosmos-Predict2.5 — Synthetic Scenario Generation")
print("  Uses the same Cosmos environment (scripts/setup_cosmos_env.sh).")
print("  Generates predicted future frames / edge-case scenarios.")
print("  Instance: ml.p4d.24xlarge")

In [ ]:
"""Mark this module complete on the participant dashboard (best-effort, non-fatal)."""
import sys
from pathlib import Path
for _b in (Path.cwd(), Path.cwd().parent, Path.home()):
    _cand = _b / "scripts" / "av30_progress.py"
    if _cand.exists():
        sys.path.insert(0, str(_b / "scripts"))
        break
try:
    from av30_progress import mark_complete
    mark_complete("m04-cosmos-transfer")
except Exception as _e:
    print(f"[progress] helper unavailable ({_e}); skipping — module still complete.")